# 第 9 天 · 结构化输出

> 📓 **这是第 9 天 的 Notebook 版**，内容和[课程网页](https://pollux1314.github.io/agent-course/#/day/9)一样。
> 先运行第一格「运行环境准备」，再从上往下逐格 Shift+Enter。你改的代码自动存在这个浏览器里；想恢复原样，在左侧文件列表右键这个文件 → Delete，刷新后会重新出现原版。
>
> 浏览器里的 Python（Pyodide）的限制：`input()`、`subprocess`、本地文件和数据库要用本地 `uv run`；访问外部网站需要对方允许跨域（DeepSeek、通义、GitHub API、httpbin 都可以）。本地脚本里的 `asyncio.run(main())` 在这里已改成 `await main()`；输出里看到 `SystemExit: ...` 是脚本主动退出（比如没填 Key），不是环境坏了。

In [ ]:
# ⚙️ 运行环境准备：打开 Notebook 后先运行这一格（Shift+Enter）
%pip install -q openai pydantic python-dotenv
import sys
sys.argv = ["main.py"]  # 课程脚本里读命令行参数的地方走默认值
try:  # 同步 httpx / openai 在浏览器里走同步 XHR（和课程网页的运行器一致）
    import httpx._transports.jsfetch as _jf
    _jf.can_run_sync = lambda: False
except Exception:
    pass
print("✓ 环境就绪，可以往下运行了")

In [ ]:
# 🔑 这一课要调用大模型：把 Key 填在引号里再运行这一格（只存在这次打开的浏览器内存里，别分享这个 Notebook）
import os
LLM_API_KEY = ""
os.environ["LLM_BASE_URL"] = "https://api.deepseek.com"   # 通义：https://dashscope.aliyuncs.com/compatible-mode/v1
os.environ["LLM_MODEL"] = "deepseek-chat"                 # 通义：qwen-plus
if LLM_API_KEY:
    os.environ["LLM_API_KEY"] = LLM_API_KEY
print("Key：", "已设置" if os.getenv("LLM_API_KEY") else "未设置（会走缺 Key 的报错分支）")

In [ ]:
"""第 9 天 · 结构化输出：从简历文本里抽取 姓名 / 技能 / 年限，输出合法 JSON
没有 Key 也能跑（假模型故意先犯错，演示"校验失败 → 把错误发回去 → 重试"）；
填了 LLM_API_KEY 就调用真模型（DeepSeek / 通义都支持 JSON mode）。"""
import asyncio
import json
import os
import re

from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError, field_validator

load_dotenv()


# ---------- 1. 用 pydantic 定义 schema（类比 Java 的 DTO + Bean Validation） ----------
class Skill(BaseModel):
    name: str = Field(description="技能名，如 Java、Spring Boot")
    years: float | None = Field(default=None, ge=0, le=50, description="该技能使用年限，不知道就填 null")


class Resume(BaseModel):
    name: str = Field(min_length=1, description="姓名")
    total_years: float = Field(ge=0, le=50, description="总工作年限")
    skills: list[Skill] = Field(min_length=1, description="技能列表")
    email: str | None = Field(default=None, description="邮箱，没有就填 null")

    @field_validator("email")
    @classmethod
    def check_email(cls, v: str | None) -> str | None:
        if v is not None and "@" not in v:
            raise ValueError("邮箱格式不对")
        return v


# 把 schema 直接塞进 prompt：模型照着这个 JSON Schema 输出
SCHEMA_JSON = json.dumps(Resume.model_json_schema(), ensure_ascii=False)

SYSTEM = f"""你是简历信息抽取器。只输出一个 JSON 对象，不要解释，不要用 ``` 包裹。
JSON 必须符合下面的 JSON Schema（数字就写数字，不要写成 "5年" 这种字符串）：
<schema>
{SCHEMA_JSON}
</schema>"""

RESUME_TEXT = """
王小明，男，邮箱 xiaoming.wang@example.com。2018 年毕业后一直做后端，至今约 8 年工作经验。
精通 Java（8 年）和 Spring Boot（6 年），熟悉 MySQL、Redis，近一年开始用 Python 写脚本，
了解 Go。目前在一家电商公司负责订单系统。
"""


# ---------- 2. 解析 + 校验：先容错地拿到 JSON，再交给 pydantic ----------
def extract_json(text: str) -> str:
    """模型有时会包一层 ```json ... ```，先剥掉"""
    m = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()


def parse(text: str) -> Resume:
    return Resume.model_validate_json(extract_json(text))


# ---------- 3. 模型：有 Key 用真的，没 Key 用假的 ----------
FAKE_ANSWERS = [
    # 第 1 次：故意犯两个常见错误——年限写成字符串、技能写成字符串列表
    '```json\n{"name": "王小明", "total_years": "8年", "skills": ["Java", "Spring Boot"]}\n```',
    # 第 2 次：看到错误信息后改对了
    '{"name": "王小明", "total_years": 8, "email": "xiaoming.wang@example.com", '
    '"skills": [{"name": "Java", "years": 8}, {"name": "Spring Boot", "years": 6}, '
    '{"name": "MySQL", "years": null}, {"name": "Redis", "years": null}, '
    '{"name": "Python", "years": 1}, {"name": "Go", "years": null}]}',
]


async def fake_llm(messages: list[dict]) -> str:
    await asyncio.sleep(0.05)
    tries = sum(1 for m in messages if m["role"] == "assistant")  # 已经回答过几次
    return FAKE_ANSWERS[min(tries, len(FAKE_ANSWERS) - 1)]


async def real_llm(messages: list[dict]) -> str:
    from openai import AsyncOpenAI
    client = AsyncOpenAI(api_key=os.environ["LLM_API_KEY"],
                         base_url=os.getenv("LLM_BASE_URL", "https://api.deepseek.com"))
    resp = await client.chat.completions.create(
        model=os.getenv("LLM_MODEL", "deepseek-chat"),
        messages=messages,
        temperature=0,
        response_format={"type": "json_object"},  # JSON mode：保证是合法 JSON（但不保证字段对）
        max_tokens=800,
    )
    return resp.choices[0].message.content or ""


# ---------- 4. 校验失败就把错误发回去，让模型自己改（最多 3 次） ----------
async def extract_resume(text: str, llm, max_retries: int = 3) -> Resume:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"<resume>\n{text.strip()}\n</resume>"},
    ]
    for attempt in range(1, max_retries + 1):
        answer = await llm(messages)
        print(f"--- 第 {attempt} 次回答 ---\n{answer}")
        try:
            resume = parse(answer)
            print(f"✅ 校验通过（第 {attempt} 次）")
            return resume
        except (ValidationError, json.JSONDecodeError) as e:
            errors = e.errors() if isinstance(e, ValidationError) else [{"msg": str(e)}]
            brief = "; ".join(f"{'.'.join(map(str, err.get('loc', [])))}: {err['msg']}" for err in errors)
            print(f"❌ 校验失败：{brief}")
            # 关键：把模型的错误回答和错误原因都放回对话，让它"看着报错改代码"
            messages.append({"role": "assistant", "content": answer})
            messages.append({"role": "user", "content":
                             f"你的 JSON 没有通过校验：{brief}\n请修正后重新输出完整 JSON，只输出 JSON。"})
    raise RuntimeError(f"重试 {max_retries} 次仍然失败")


async def main():
    llm = real_llm if os.getenv("LLM_API_KEY") else fake_llm
    print("使用", "真模型 " + os.getenv("LLM_MODEL", "deepseek-chat") if llm is real_llm else "假模型（没填 Key）")
    resume = await extract_resume(RESUME_TEXT, llm)
    print("\n===== 拿到的是 Python 对象，不是字符串 =====")
    print("姓名:", resume.name, "| 总年限:", resume.total_years)
    for s in resume.skills:
        print(f"  - {s.name}: {s.years if s.years is not None else '未知'} 年")
    print("\n最终 JSON：")
    print(resume.model_dump_json(indent=2))


if __name__ == "__main__":
    await main()  # 本地脚本写 asyncio.run(...)，Notebook 里直接 await